In [35]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from pandas.api.types import CategoricalDtype

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)
from sklearn.model_selection import train_test_split

In [36]:
df = pd.read_csv("../data/raw/hospital_readmissions.csv")


# Definir Target com foco (readmitted)

In [37]:
df = pd.read_csv("../data/raw/hospital_readmissions.csv")
y = df['readmitted']

# Definir Features 

In [38]:
x = df.drop('readmitted', axis=1)


In [39]:
x.shape

(25000, 16)

In [40]:
x.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   age                25000 non-null  str  
 1   time_in_hospital   25000 non-null  int64
 2   n_lab_procedures   25000 non-null  int64
 3   n_procedures       25000 non-null  int64
 4   n_medications      25000 non-null  int64
 5   n_outpatient       25000 non-null  int64
 6   n_inpatient        25000 non-null  int64
 7   n_emergency        25000 non-null  int64
 8   medical_specialty  25000 non-null  str  
 9   diag_1             25000 non-null  str  
 10  diag_2             25000 non-null  str  
 11  diag_3             25000 non-null  str  
 12  glucose_test       25000 non-null  str  
 13  A1Ctest            25000 non-null  str  
 14  change             25000 non-null  str  
 15  diabetes_med       25000 non-null  str  
dtypes: int64(7), str(9)
memory usage: 3.1 MB


In [41]:
numerical_features = ['time_in_hospital', 'n_lab_procedures', 'n_procedures', 'n_medications', 'n_outpatient', 'n_inpatient', 'n_emergency']
categorical_features = ['age', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'glucose_test', 'A1Ctest', 'change', 'diabetes_med']

# Identificação das variáveis

As variáveis foram separadas em numéricas e categóricas para permitir a aplicação de técnicas de pré-processamento específicas para cada tipo de dado. Variáveis numéricas serão mantidas em sua representação original, enquanto as categóricas serão posteriormente codificadas para utilização pelos algoritmos de Machine Learning.

In [42]:
y.shape

(25000,)

In [43]:
print(sorted(df['age'].unique()))

['[40-50)', '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']


In [ ]:
df = pd.read_csv("../data/raw/hospital_readmissions.csv")

X = df.drop(columns='readmitted')
y = df['readmitted']

numerical_features = [
    'time_in_hospital',
    'n_lab_procedures',
    'n_procedures',
    'n_medications',
    'n_outpatient',
    'n_inpatient',
    'n_emergency'
]

ordinal_features = [
    'age',
    'glucose_test',
    'A1Ctest',
    'change',
    'diabetes_med'
]

onehot_features = [
    'medical_specialty',
    'diag_1',
    'diag_2',
    'diag_3'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),

        ('ord',
         OrdinalEncoder(categories=[
             ['[40-50)', '[50-60)', '[60-70)',
              '[70-80)', '[80-90)', '[90-100)'],
             ['no', 'normal', 'high'],
             ['no', 'normal', 'high'],
             ['no', 'yes'],
             ['no', 'yes']]),ordinal_features),

        ('cat',
         OneHotEncoder(
             handle_unknown='ignore',
             sparse_output=False),onehot_features)],remainder='drop')




age:

glucose_test:

A1Ctest:

change:

diabetes_med:


# Identificação das Features
Numerical Features: variáveis quantitativas que serão utilizadas em sua escala original (com possibilidade de padronização posteriormente).
Ordinal Features: variáveis categóricas que apresentam uma ordem natural entre suas categorias.
One-Hot Features: variáveis categóricas nominais, sem relação hierárquica entre seus valores.

## Train-Test Split

Nesta etapa, o dataset é dividido em conjuntos de treino e teste.
O conjunto de treino será utilizado para ajustar o pipeline de pré-processamento e treinar os modelos, enquanto o conjunto de teste permanecerá isolado para avaliar o desempenho em dados nunca vistos.

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [51]:
print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test: {y_test.shape}')


X_train: (20000, 16)
X_test: (5000, 16)
y_train: (20000,)
y_test: (5000,)


In [52]:
print("Distribuição original:")
print(y.value_counts(normalize=True) * 100)

print("\nTreino:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTeste:")
print(y_test.value_counts(normalize=True) * 100)

Distribuição original:
readmitted
no     52.984
yes    47.016
Name: proportion, dtype: float64

Treino:
readmitted
no     52.985
yes    47.015
Name: proportion, dtype: float64

Teste:
readmitted
no     52.98
yes    47.02
Name: proportion, dtype: float64


In [53]:
x_train_processed = preprocessor.fit_transform(X_train)
x_test_processed = preprocessor.transform(X_test)

In [55]:
print(x_train_processed.shape)
print(x_test_processed.shape)

(20000, 43)
(5000, 43)


In [56]:
feature_names = preprocessor.get_feature_names_out()

print(f'Total de features após o pré-processamento: {len(feature_names)}')


Total de features após o pré-processamento: 43


In [57]:
feature_names[:20]

array(['num__time_in_hospital', 'num__n_lab_procedures',
       'num__n_procedures', 'num__n_medications', 'num__n_outpatient',
       'num__n_inpatient', 'num__n_emergency', 'ord__age',
       'ord__glucose_test', 'ord__A1Ctest', 'ord__change',
       'ord__diabetes_med', 'cat__medical_specialty_Cardiology',
       'cat__medical_specialty_Emergency/Trauma',
       'cat__medical_specialty_Family/GeneralPractice',
       'cat__medical_specialty_InternalMedicine',
       'cat__medical_specialty_Missing', 'cat__medical_specialty_Other',
       'cat__medical_specialty_Surgery', 'cat__diag_1_Circulatory'],
      dtype=object)